## Transfer Learning with Segmentation Models

In [ ]:
import segmentation_models as sm
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from wandb.keras import WandbCallback

from networks import *

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
HEIGHT = 256
WIDTH = 256
CHANNELS = 3
IMAGE_SHAPE = (HEIGHT, WIDTH, CHANNELS)

EPOCHS = 50
STEPS_PER_EPOCH = 1000
BATCH_SIZE = 8
DEEP_SUPERVISION = False
LOGGING = False

BACKBONE = 'resnet34'
preprocess_input = sm.get_preprocessing(BACKBONE)

In [ ]:
# Load dataset
dataset_path = 'input/'
train_ds, val_ds, test_ds = load_dataset(
    dataset_path, val_split=0.1, batch_size=BATCH_SIZE,
    height=HEIGHT, width=WIDTH, channels=CHANNELS,
    preprocess=preprocess_input, augment=True, repeat=True,
)

In [ ]:
model = sm.Unet(
    BACKBONE,
    encoder_weights='imagenet',
    input_shape=IMAGE_SHAPE,
    classes=NUM_CLASSES,
    activation='softmax',
    encoder_freeze=True,
)

model.compile(
    optimizer='Adam',
    loss=sm.losses.bce_jaccard_loss,
    metrics=['accuracy', jaccard_coef, dice_coef, sm.metrics.iou_score, sm.metrics.f1_score],
)

In [ ]:
# Smaller dataset for testing
EPOCHS = 3
STEPS_PER_EPOCH = 2
train_ds = train_ds.take(STEPS_PER_EPOCH)
val_ds = val_ds.take(STEPS_PER_EPOCH)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, verbose=1),
    VisualizeProgress(
        test_data=next(iter(val_ds)), every_n_epochs=1, num_samples=5,
        convert_mask=one_hot_to_rgb, deep_supervision=DEEP_SUPERVISION, log=LOGGING,
    ),
]
if LOGGING:
    callbacks.append(WandbCallback())

history = model.fit(
    train_ds, validation_data=val_ds, validation_steps=len(val_ds),
    epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks, verbose=1,
)

In [ ]:
model_name = f'models/sm-unet-{BACKBONE}.h5'
model.save(model_name)

In [ ]:
model = tf.keras.models.load_model(model_name, compile=False)
model.compile(
    optimizer='Adam',
    loss=sm.losses.bce_jaccard_loss,
    metrics=[sm.metrics.iou_score, sm.metrics.f1_score]
)
model.evaluate(test_ds, steps=len(test_ds))